In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train=pd.read_csv('/kaggle/input/playground-series-s6e2/train.csv')
test=pd.read_csv('/kaggle/input/playground-series-s6e2/test.csv')

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
test.describe()

In [ ]:
train.info()

In [ ]:
test.info()

In [ ]:
test.isnull().sum()

# EDA

In [ ]:
import matplotlib.pyplot as plt
target_counts = train['Heart Disease'].value_counts()
print(target_counts)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

target_counts.plot(kind='bar', ax=axes[0], color=['#2196F3','#F44336'], edgecolor='black')
axes[0].set_title('Heart Disease Class Distribution', fontsize=13)
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{int(bar.get_height()):,}', ha='center', fontsize=10)

axes[1].pie(target_counts, labels=target_counts.index, autopct='%1.1f%%',
            colors=['#2196F3','#F44336'], startangle=90)
axes[1].set_title('Class Proportion', fontsize=13)

plt.tight_layout()
plt.show()

In [ ]:
# numeric feature distributions split by target
import seaborn as sns
num_cols = ['Age','BP','Cholesterol','Max HR','ST depression']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    for label, color in [('Absence','#2196F3'), ('Presence','#F44336')]:
        subset = train[train['Heart Disease'] == label][col]
        axes[i].hist(subset, bins=30, alpha=0.6, color=color, label=label, edgecolor='white')
    axes[i].set_title(col, fontsize=12)
    axes[i].legend()
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')

# Feature correlation heatmap in last subplot
feature_cols_eda = [c for c in train.columns if c not in ['id','Heart Disease']]
corr = train[feature_cols_eda].corr()
sns.heatmap(corr, ax=axes[5], cmap='coolwarm', center=0,
            linewidths=0.5, annot=False, fmt='.1f')
axes[5].set_title('Feature Correlation Heatmap', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Categorical feature counts by target
cat_cols = ['Sex','Chest pain type','FBS over 120','EKG results',
            'Exercise angina','Slope of ST','Number of vessels fluro','Thallium']

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ct = pd.crosstab(train[col], train['Heart Disease'])
    ct.plot(kind='bar', ax=axes[i], color=['#2196F3','#F44336'],
            edgecolor='black', alpha=0.85)
    axes[i].set_title(col, fontsize=11)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].legend(title='Heart Disease', fontsize=8)

plt.tight_layout()
plt.show()


# Feature engineering

In [ ]:
def add_features(df):
    df = df.copy()
    df['age_hr_ratio']    = df['Age'] / (df['Max HR'] + 1)
    df['bp_chol_ratio']   = df['BP'] / (df['Cholesterol'] + 1)
    df['st_slope_inter']  = df['ST depression'] * df['Slope of ST']
    df['age_bp']          = df['Age'] * df['BP']
    df['thallium_high']   = (df['Thallium'] == 7).astype(int)
    df['vessels_nonzero'] = (df['Number of vessels fluro'] > 0).astype(int)
    df['hr_age_diff']     = df['Max HR'] - df['Age']          # proxy for cardio fitness
    df['cp_angina']       = df['Chest pain type'] * df['Exercise angina']
    return df

feature_cols = [c for c in train.columns if c not in ['id', 'Heart Disease']]

X      = add_features(train[feature_cols])
y      = train['Heart Disease'].map({'Absence': 0, 'Presence': 1})
X_test = add_features(test[feature_cols])
test_ids = test['id']

print(f"Features after engineering: {X.shape[1]}  (was {len(feature_cols)})")
print("New features:", [c for c in X.columns if c not in feature_cols])


# K fold CV

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, log_loss

N_SPLITS   = 5
RANDOM_STATE = 42

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# Storage for OOF predictions from each model
oof_xgb  = np.zeros(len(X))
oof_lgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))

# Storage for test predictions (averaged across folds)
test_xgb = np.zeros(len(X_test))
test_lgb = np.zeros(len(X_test))
test_cat = np.zeros(len(X_test))

# Class imbalance weight
neg = (y == 0).sum()
pos = (y == 1).sum()
spw = neg / pos
print(f"Positive class: {pos:,}  |  Negative class: {neg:,}  |  scale_pos_weight ≈ {spw:.3f}")

# XGBOOST

In [ ]:
from xgboost import XGBClassifier
from sklearn.base import clone

xgb_params = dict(
    n_estimators          = 2000,
    max_depth             = 5,
    learning_rate         = 0.03,
    subsample             = 0.80,
    colsample_bytree      = 0.80,
    min_child_weight      = 5,
    gamma                 = 0.1,
    reg_alpha             = 0.1,
    reg_lambda            = 1.0,
    scale_pos_weight      = spw,
    grow_policy           = 'depthwise',
    eval_metric           = 'auc',
    early_stopping_rounds = 30,
    tree_method           = 'hist',
    n_jobs                = -1,
    random_state          = RANDOM_STATE,
)

xgb_base = XGBClassifier(**xgb_params)

xgb_scores_auc  = []
xgb_scores_loss = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_va_s = sc.transform(X_va)
    X_te_s = sc.transform(X_test)

    clf = clone(xgb_base)
    clf.fit(X_tr_s, y_tr, eval_set=[(X_va_s, y_va)], verbose=False)

    p_va = clf.predict_proba(X_va_s)[:, 1]
    oof_xgb[va_idx] = p_va
    test_xgb += clf.predict_proba(X_te_s)[:, 1] / N_SPLITS

    auc  = roc_auc_score(y_va, p_va)
    loss = log_loss(y_va, p_va)
    xgb_scores_auc.append(auc)
    xgb_scores_loss.append(loss)
    print(f"  Fold {fold}  AUC={auc:.5f}  LogLoss={loss:.5f}  best_iter={clf.best_iteration}")

print(f"\nXGBoost CV  AUC: {np.mean(xgb_scores_auc):.5f} ± {np.std(xgb_scores_auc):.5f}"
      f"   LogLoss: {np.mean(xgb_scores_loss):.5f} ± {np.std(xgb_scores_loss):.5f}")

# LIGHT GBM

In [ ]:
from lightgbm import LGBMClassifier

lgb_params = dict(
    n_estimators      = 2000,
    num_leaves        = 31,
    max_depth         = -1,
    learning_rate     = 0.03,
    subsample         = 0.80,
    colsample_bytree  = 0.80,
    min_child_samples = 20,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    scale_pos_weight  = spw,
    n_jobs            = -1,
    random_state      = RANDOM_STATE,
    verbose           = -1,
)

lgb_base = LGBMClassifier(**lgb_params)

lgb_scores_auc  = []
lgb_scores_loss = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_va_s = sc.transform(X_va)
    X_te_s = sc.transform(X_test)

    clf = clone(lgb_base)
    clf.fit(X_tr_s, y_tr,
            eval_set=[(X_va_s, y_va)],
            callbacks=[
                __import__('lightgbm').early_stopping(30, verbose=False),
                __import__('lightgbm').log_evaluation(-1)
            ])

    p_va = clf.predict_proba(X_va_s)[:, 1]
    oof_lgb[va_idx] = p_va
    test_lgb += clf.predict_proba(X_te_s)[:, 1] / N_SPLITS

    auc  = roc_auc_score(y_va, p_va)
    loss = log_loss(y_va, p_va)
    lgb_scores_auc.append(auc)
    lgb_scores_loss.append(loss)
    print(f"  Fold {fold}  AUC={auc:.5f}  LogLoss={loss:.5f}  best_iter={clf.best_iteration_}")

print(f"\nLightGBM CV AUC: {np.mean(lgb_scores_auc):.5f} ± {np.std(lgb_scores_auc):.5f}"
      f"   LogLoss: {np.mean(lgb_scores_loss):.5f} ± {np.std(lgb_scores_loss):.5f}")

In [ ]:
from catboost import CatBoostClassifier

cat_params = dict(
    iterations        = 2000,
    depth             = 6,
    learning_rate     = 0.03,
    subsample         = 0.80,
    colsample_bylevel = 0.80,
    l2_leaf_reg       = 3.0,
    class_weights     = [1.0, spw],   # CatBoost uses class_weights
    eval_metric       = 'AUC',
    early_stopping_rounds = 30,
    random_seed       = RANDOM_STATE,
    verbose           = 0,
)

cat_base = CatBoostClassifier(**cat_params)

cat_scores_auc  = []
cat_scores_loss = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    # CatBoost doesn't need scaling, but we keep it consistent
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_va_s = sc.transform(X_va)
    X_te_s = sc.transform(X_test)

    clf = CatBoostClassifier(**cat_params)
    clf.fit(X_tr_s, y_tr,
            eval_set=(X_va_s, y_va),
            use_best_model=True,
            verbose=0)

    p_va = clf.predict_proba(X_va_s)[:, 1]
    oof_cat[va_idx] = p_va
    test_cat += clf.predict_proba(X_te_s)[:, 1] / N_SPLITS

    auc  = roc_auc_score(y_va, p_va)
    loss = log_loss(y_va, p_va)
    cat_scores_auc.append(auc)
    cat_scores_loss.append(loss)
    print(f"  Fold {fold}  AUC={auc:.5f}  LogLoss={loss:.5f}  best_iter={clf.get_best_iteration()}")

print(f"\nCatBoost CV  AUC: {np.mean(cat_scores_auc):.5f} ± {np.std(cat_scores_auc):.5f}"
      f"   LogLoss: {np.mean(cat_scores_loss):.5f} ± {np.std(cat_scores_loss):.5f}")

# Model Comparison

In [ ]:
from sklearn.metrics import roc_curve

# OOF metrics (whole dataset, no fold averaging)
oof_auc_xgb  = roc_auc_score(y, oof_xgb)
oof_auc_lgb  = roc_auc_score(y, oof_lgb)
oof_auc_cat  = roc_auc_score(y, oof_cat)

oof_loss_xgb = log_loss(y, oof_xgb)
oof_loss_lgb = log_loss(y, oof_lgb)
oof_loss_cat = log_loss(y, oof_cat)

results = pd.DataFrame({
    'Model'   : ['XGBoost',    'LightGBM',   'CatBoost'],
    'OOF AUC' : [oof_auc_xgb,  oof_auc_lgb,  oof_auc_cat],
    'OOF LogLoss': [oof_loss_xgb, oof_loss_lgb, oof_loss_cat],
    'CV AUC mean' : [np.mean(xgb_scores_auc), np.mean(lgb_scores_auc), np.mean(cat_scores_auc)],
    'CV AUC std'  : [np.std(xgb_scores_auc),  np.std(lgb_scores_auc),  np.std(cat_scores_auc)],
})
print(results.to_string(index=False))


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2196F3', '#4CAF50', '#FF5722']
for oof, label, color in [(oof_xgb,'XGBoost',colors[0]),
                           (oof_lgb,'LightGBM',colors[1]),
                           (oof_cat,'CatBoost',colors[2])]:
    fpr, tpr, _ = roc_curve(y, oof)
    auc = roc_auc_score(y, oof)
    axes[0].plot(fpr, tpr, label=f'{label} (AUC={auc:.4f})', color=color, lw=2)
axes[0].plot([0,1],[0,1],'k--', lw=1)
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('ROC Curves – OOF Predictions', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)


x   = np.arange(3)
w   = 0.35
models = ['XGBoost', 'LightGBM', 'CatBoost']
aucs   = [oof_auc_xgb, oof_auc_lgb, oof_auc_cat]
losses = [oof_loss_xgb, oof_loss_lgb, oof_loss_cat]

ax2 = axes[1]
bars1 = ax2.bar(x - w/2, aucs,   width=w, label='OOF AUC (↑)',     color='#2196F3', alpha=0.85)
ax2r  = ax2.twinx()
bars2 = ax2r.bar(x + w/2, losses, width=w, label='OOF LogLoss (↓)', color='#F44336', alpha=0.85)

ax2.set_xticks(x); ax2.set_xticklabels(models, fontsize=11)
ax2.set_ylabel('AUC', fontsize=12, color='#2196F3')
ax2r.set_ylabel('Log-Loss', fontsize=12, color='#F44336')
ax2.set_title('AUC vs Log-Loss per Model', fontsize=13)
ax2.set_ylim(0.9, 1.0)
ax2r.set_ylim(0, max(losses)*1.5)

lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2r.get_legend_handles_labels()
ax2.legend(lines1+lines2, labels1+labels2, loc='upper right', fontsize=10)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
from scipy.optimize import minimize

oof_avg  = (oof_xgb + oof_lgb + oof_cat) / 3
test_avg = (test_xgb + test_lgb + test_cat) / 3
auc_avg  = roc_auc_score(y, oof_avg)
loss_avg = log_loss(y, oof_avg)
print(f"Simple Average  ── AUC: {auc_avg:.5f}   LogLoss: {loss_avg:.5f}")

from scipy.stats import rankdata
def rank_avg(preds_list):
    ranks = np.column_stack([rankdata(p) / len(p) for p in preds_list])
    return ranks.mean(axis=1)

oof_rank  = rank_avg([oof_xgb, oof_lgb, oof_cat])
test_rank = rank_avg([test_xgb, test_lgb, test_cat])
auc_rank  = roc_auc_score(y, oof_rank)
loss_rank = log_loss(y, oof_rank)
print(f"Rank Average    ── AUC: {auc_rank:.5f}   LogLoss: {loss_rank:.5f}")

def neg_auc(weights):
    w = np.array(weights)
    w = np.clip(w, 0, 1)
    w /= w.sum()
    blend = w[0]*oof_xgb + w[1]*oof_lgb + w[2]*oof_cat
    return -roc_auc_score(y, blend)

result = minimize(neg_auc, x0=[1/3, 1/3, 1/3],
                  method='Nelder-Mead',
                  options={'maxiter': 1000, 'xatol': 1e-6})

opt_w = np.clip(result.x, 0, 1)
opt_w /= opt_w.sum()
print(f"\nOptimal weights ── XGB: {opt_w[0]:.3f}  LGB: {opt_w[1]:.3f}  CAT: {opt_w[2]:.3f}")

oof_opt  = opt_w[0]*oof_xgb  + opt_w[1]*oof_lgb  + opt_w[2]*oof_cat
test_opt = opt_w[0]*test_xgb + opt_w[1]*test_lgb + opt_w[2]*test_cat
auc_opt  = roc_auc_score(y, oof_opt)
loss_opt = log_loss(y, oof_opt)
print(f"Optimized Blend ── AUC: {auc_opt:.5f}   LogLoss: {loss_opt:.5f}")

summary = pd.DataFrame({
    'Method'  : ['XGBoost','LightGBM','CatBoost','Simple Avg','Rank Avg','Opt Blend'],
    'OOF AUC' : [oof_auc_xgb, oof_auc_lgb, oof_auc_cat, auc_avg, auc_rank, auc_opt],
    'OOF LogLoss': [oof_loss_xgb, oof_loss_lgb, oof_loss_cat, loss_avg, loss_rank, loss_opt],
})
summary = summary.sort_values('OOF AUC', ascending=False).reset_index(drop=True)
print("\n── Final Comparison ──")
print(summary.to_string(index=False))

In [ ]:
# visual comparison of all ensemble strategies
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

methods  = summary['Method'].tolist()
auc_vals = summary['OOF AUC'].tolist()
loss_vals= summary['OOF LogLoss'].tolist()
bar_colors = ['#FF5722' if m == summary['Method'].iloc[0] else '#90CAF9' for m in methods]

axes[0].barh(methods[::-1], auc_vals[::-1], color=bar_colors[::-1], edgecolor='black', alpha=0.9)
axes[0].set_xlabel('OOF AUC', fontsize=12)
axes[0].set_title('OOF AUC by Method (higher = better)', fontsize=13)
axes[0].set_xlim(min(auc_vals)*0.999, 1.0)
for i, v in enumerate(auc_vals[::-1]):
    axes[0].text(v + 0.00005, i, f'{v:.5f}', va='center', fontsize=10)
axes[0].grid(axis='x', alpha=0.3)

axes[1].barh(methods[::-1], loss_vals[::-1], color=bar_colors[::-1], edgecolor='black', alpha=0.9)
axes[1].set_xlabel('OOF Log-Loss', fontsize=12)
axes[1].set_title('OOF Log-Loss by Method (lower = better)', fontsize=13)
for i, v in enumerate(loss_vals[::-1]):
    axes[1].text(v + 0.00005, i, f'{v:.5f}', va='center', fontsize=10)
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('Ensemble Strategy Comparison', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Use the best ensemble (optimized blend)
sample_sub = pd.read_csv('/kaggle/input/playground-series-s6e2/sample_submission.csv')
target_col = [c for c in sample_sub.columns if c != 'id'][0]
print(f"Submission target column: '{target_col}'")

submission = pd.DataFrame({'id': test_ids, target_col: test_opt})
submission.to_csv('submission.csv', index=False)

print(f"\nSubmission saved with {len(submission):,} rows.")
print("Preview:")
print(submission.head(10))

# Prediction distribution sanity check
plt.figure(figsize=(8, 4))
plt.hist(test_opt, bins=50, color='#2196F3', edgecolor='white', alpha=0.85)
plt.axvline(0.5, color='red', linestyle='--', label='Decision threshold (0.5)')
plt.xlabel('Predicted Probability of Heart Disease', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Distribution of Test Predictions', fontsize=13)
plt.legend()
plt.tight_layout()
plt.show()